# PoC step 5: mock gateway and live demo
Starts the FastAPI mock wallet and hub, runs the demo scenarios, and replays all 402 bank records through the hub.

**Before you start:** copy `mock_core.py`, `mock_gateway.py`, `test_mock_core.py`, and `test_mock_gateway_api.py` into `bw-interop-poc/src`, and replace `README.md` and `requirements.txt`. Steps 1 to 4 must have run (the data files exist in `data`).

Everything here is a simulation with synthetic data. The wallet is a mock, not Orange's API.

In [1]:
# Cell 1: mount Drive and install the web dependencies
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/bw-interop-poc'
!pip -q install fastapi uvicorn httpx requests pytest "pydantic>=2" pyyaml

Mounted at /content/drive


In [2]:
# Cell 2: run all tests (expect 93 passed: 58 earlier + 29 mock core + 6 HTTP)
!cd {BASE}/src && python -m pytest -q -p no:cacheprovider

........................................................................ [ 77%]
.....................                                                    [100%]
93 passed in 17.82s


In [3]:
# Cell 3: start the server in the background and wait until it answers
import subprocess, time, requests
server = subprocess.Popen(['python', '-m', 'uvicorn', 'mock_gateway:app', '--app-dir', f'{BASE}/src', '--port', '8000'],
                          stdout=open('/content/uvicorn.log', 'w'), stderr=subprocess.STDOUT)
URL = 'http://127.0.0.1:8000'
for _ in range(30):
    try:
        requests.get(f'{URL}/health', timeout=1)
        break
    except Exception:
        time.sleep(1)
print(requests.get(f'{URL}/health').json())

{'status': 'ok', 'mode': 'SIMULATION: mock wallet and hub, synthetic data only'}


In [4]:
# Cell 4 (optional): link to the Swagger page through Colab's proxy, for a screenshot
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)") + '/docs')

https://8000-m-s-kkb-ase1a0-177ynf2ifxwbb-a.asia-east1-0.prod.colab.dev/docs


In [5]:
# Cell 5: demo scenarios on real generated bank records
import sys
sys.path.insert(0, f'{BASE}/src')
from loader import parse_pacs008
_, records = parse_pacs008(f'{BASE}/data/poc_pacs008.xml')
requests.post(f'{URL}/mock/reset')

def post(rec, simulate='SUCCESS'):
    return requests.post(f'{URL}/interop/v1/bank-to-wallet', params={'simulate': simulate}, json=rec)

def brief(r):
    body = r.json()
    if r.status_code == 200:
        return {'http': 200, 'e2e': body['end_to_end_id'], 'wallet_status': body['wallet_order']['status'], 'canonical': body['canonical_status'],
                'iso': body['iso_status'], 'notification_verified': body['notification_verified']}
    detail = body.get('detail')
    return {'http': r.status_code, 'reasons': [x['code'] for x in detail]} if isinstance(detail, list) else {'http': r.status_code, 'detail': detail}

valid = [r for r in records if r['currency'] and r['creditor_bic'] == 'MOCKBWOMXXX'][:3]
missing = next(r for r in records if not r['currency'])
print('1 success                  ', brief(post(valid[0])))
print('2 same record again        ', brief(post(valid[0])))
print('3 missing currency         ', brief(post(missing)))
print('4 wallet payment fails     ', brief(post(valid[1], 'FAILED')))
print('5 customer has not paid yet', brief(post(valid[2], 'PENDING')))
print('6 look up payment 1        ', brief(requests.get(f'{URL}/interop/v1/payments/{valid[0]["end_to_end_id"]}')))
print()
print('The wallet request the hub built for payment 1:')
import json
print(json.dumps(requests.get(f'{URL}/interop/v1/payments/{valid[0]["end_to_end_id"]}').json()['wallet_request'], indent=2))

1 success                   {'http': 200, 'e2e': 'E2E-BW-20260807-0001', 'wallet_status': 'SUCCESS', 'canonical': 'COMPLETED', 'iso': 'ACSC', 'notification_verified': True}
2 same record again         {'http': 409, 'reasons': ['DUPLICATE_KEY']}
3 missing currency          {'http': 422, 'reasons': ['MISSING_CURRENCY']}
4 wallet payment fails      {'http': 200, 'e2e': 'E2E-BW-20260830-0002', 'wallet_status': 'FAILED', 'canonical': 'FAILED', 'iso': 'RJCT', 'notification_verified': True}
5 customer has not paid yet {'http': 200, 'e2e': 'E2E-BW-20260810-0003', 'wallet_status': 'INITIATED', 'canonical': 'DISPATCHED', 'iso': 'PDNG', 'notification_verified': None}
6 look up payment 1         {'http': 200, 'e2e': 'E2E-BW-20260807-0001', 'wallet_status': 'SUCCESS', 'canonical': 'COMPLETED', 'iso': 'ACSC', 'notification_verified': True}

The wallet request the hub built for payment 1:
{
  "order_id": "E2E-BW-20260807-0001",
  "reference": "Invoice 2026-0001",
  "amount": {
    "value": 18521.71,


In [6]:
# Cell 6: the wallet side on its own: token, payment, customer pays, status, notifications
import base64
basic = 'Basic ' + base64.b64encode(b'mock-client:mock-secret').decode()
tok = requests.post(f'{URL}/oauth/v3/token', headers={'Authorization': basic}).json()
h = {'Authorization': 'Bearer ' + tok['access_token']}
order = {'merchant_key': 'MOCK-MERCHANT-KEY', 'currency': 'BWP', 'order_id': 'DEMO-ORDER-1', 'amount': 125.5, 'return_url': 'https://example.invalid/return',
         'cancel_url': 'https://example.invalid/cancel', 'notif_url': 'https://example.invalid/notify', 'lang': 'en', 'reference': 'Demo'}
created = requests.post(f'{URL}/orange-money-webpay/dev/v1/webpayment', json=order, headers=h)
print('create payment  ', created.status_code, created.json())
pay_token = created.json()['pay_token']
print('customer pays   ', requests.post(f'{URL}/mock/customer/pay/{pay_token}', json={'outcome': 'SUCCESS'}).json())
print('status          ', requests.post(f'{URL}/orange-money-webpay/dev/v1/transactionstatus',
                                        json={'order_id': 'DEMO-ORDER-1', 'amount': 125.5, 'pay_token': pay_token}, headers=h).json())
print('wrong merchant  ', requests.post(f'{URL}/orange-money-webpay/dev/v1/webpayment', json={**order, 'order_id': 'DEMO-ORDER-2', 'merchant_key': 'bad'}, headers=h).json())
print('duplicate order ', requests.post(f'{URL}/orange-money-webpay/dev/v1/webpayment', json=order, headers=h).json())
print('notifications   ', len(requests.get(f'{URL}/mock/notifications').json()))

create payment   201 {'status': 201, 'message': 'OK', 'pay_token': 'bdb6ba02286f55a63cc18edd', 'payment_url': 'https://mock-wallet.invalid/pay/bdb6ba02286f55a63cc18edd', 'notif_token': '828ed7cf01168f6b11237ba5'}
customer pays    {'order_id': 'DEMO-ORDER-1', 'status': 'SUCCESS', 'notif_token': '828ed7cf01168f6b11237ba5', 'txnid': 'MP0A4AE28266A6', 'amount': '125.5', 'currency': 'BWP'}
status           {'status': 'SUCCESS', 'order_id': 'DEMO-ORDER-1', 'txnid': 'MP0A4AE28266A6'}
wrong merchant   {'detail': {'code': 'INVALID_MERCHANT_KEY', 'message': 'merchant key not recognised'}}
duplicate order  {'detail': {'code': 'DUPLICATE_ORDER_ID', 'message': "order_id 'DEMO-ORDER-1' already used"}}
notifications    3


In [7]:
# Cell 7: replay all 402 bank records through the hub (expect accepted 382; rejected 20)
from collections import Counter
requests.post(f'{URL}/mock/reset')
ok, codes = 0, Counter()
for rec in records:
    r = post(rec)
    if r.status_code == 200:
        ok += 1
    else:
        codes[(r.status_code, r.json()['detail'][0]['code'])] += 1
print('accepted:', ok, '| rejected:', dict(codes))
assert ok == 382 and sum(codes.values()) == 20, 'verdicts differ from the loader'

accepted: 382 | rejected: {(422, 'MISSING_CURRENCY'): 6, (409, 'DUPLICATE_KEY'): 8, (422, 'UNKNOWN_BIC'): 6}


In [8]:
# Cell 8: stop the server
server.terminate()
print('server stopped')

server stopped


## Done when
- Cell 2 ends with `93 passed`.
- Cell 3 prints the health message with the word SIMULATION.
- Cell 5 shows: 1 success COMPLETED/ACSC, 2 duplicate 409, 3 missing currency 422, 4 FAILED/RJCT, 5 PENDING/PDNG.
- Cell 7 prints `accepted: 382` and rejects 6 MISSING_CURRENCY and 6 UNKNOWN_BIC (422) plus 8 DUPLICATE_KEY (409), the same verdicts as the loader.

The HTTP layer was not run in my environment (FastAPI could not be installed there), so the six HTTP tests and Cells 3 to 7 are the first real run of the web layer. Send me any failure message and I will fix it.